In [1]:
import os
import sys
from pathlib import Path

# **Primo test Baseline**


In [2]:
#(LOCAL vs COLAB)
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    print("🌍 Detetected enviroment: COLAB")
    from google.colab import drive
    drive.mount('/content/drive')
    input_dir = Path('')
    output_dir = Path('')
else:
    print("💻 Detetected enviroment: LOCAL")
    # Impostate la vostra cartella di input e output locale
    input_dir = Path('../data/')
    output_dir = Path('../data/output')

    # Crea le cartelle se non esistono in locale
    output_dir.mkdir(parents=True, exist_ok=True)
    
INPUT_DIR, OUTPUT_DIR = input_dir, output_dir
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

💻 Detetected enviroment: LOCAL
Input directory: ../data
Output directory: ../data/output


## API milionario


In [3]:
#OCCHIO CHE VB AGGIUNGE SYS PATH QUINDI METTETE PERCORSO GIUSTO (teoricamente è giusto già)

# Define the path to the directory containing your package
package_parent_dir = './'
package_parent_dir = os.path.abspath(package_parent_dir)
#print(package_parent_dir)

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)
    

# Verify the path was added
#print(sys.path)


#import your package
from millionaire_client import MillionaireClient, AuthenticationError
import dotenv
dotenv.load_dotenv()

#credentiali per accedere al server
username = os.getenv('USER')
pwd = os.getenv('PASSWORD')
mail=os.getenv('MAIL')
API_URL = "http://131.175.15.22:51111/"

# Create a client instance
client = MillionaireClient(API_URL)
print(f"username: {username}, password: {pwd}, mail: {mail}, API_URL: {API_URL}")
try:
    # Authenticate with the server
    user = client.login(username, pwd)
    print("✅ Login successful :)")
    print(f"\n Welcome, {user.username}! Role: {user.role}")
except AuthenticationError as e:
    print(f"❌ Login failed: {e}")

username: lillaro, password: lillarowins01, mail: /var/spool/mail/lillaro, API_URL: http://131.175.15.22:51111/
✅ Login successful :)

 Welcome, lillaro! Role: student


## Local LLM Integration
Random small models
Install the extra requirements first:
  pip install -r requirements.txt

In [4]:
from transformers import pipeline
import torch

print("Loading local language model... (this may take a moment)")
llm_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen3-VL-2B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",  # GPU if available, otherwise CPU
    trust_remote_code=False,
)
print("Model loaded successfully!")

/home/lillaro/Documents/Programmazione/NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading local language model... (this may take a moment)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 625/625 [00:01<00:00, 386.60it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
[transformers] The model 'Qwen3VLForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausa

Model loaded successfully!


In [33]:
def llm_answer_strategy(question):
    """
    Ask the local LLM to pick the best option.
    Returns the *option_id* (int) so it can be passed directly to game.answer().
    """
    if not question or not question.options:
        raise ValueError("No question/options provided")

    letters = ['A', 'B', 'C', 'D'] 
    opts = question.options[:4]  # never more than 4 options
    options_text = "\n".join(
        [f"{letters[i]}: {opt.text}" for i, opt in enumerate(opts)]
    )

    prompt = (
        f"Question: {question.text}\n"
        f"Options:\n{options_text}\n"
        f"Answer with just the letter of the correct option (A, B, C, or D).\n"
        f"Answer:"
    )

    # Format as a chat so the instruct model behaves correctly
    messages = [{"role": "user", "content": prompt}]
    try:
        prompt_text = llm_pipeline.tokenizer.apply_chat_template(
            messages, tokenize=False, enable_thinking=False
        )
    except Exception:
        prompt_text = prompt  # Fallback for non-chat tokenizers

    outputs = llm_pipeline(
        prompt_text,
        max_new_tokens=5,
        do_sample=False,
        return_full_text=False,
    )

    generated = outputs[0]["generated_text"].strip().upper()
    print(f"Prompt: {prompt_text}")
    print(f"[LLM raw output] '{generated}'")

    for char in generated:
        if char in letters:
            idx = letters.index(char)
            return opts[idx].id

    print("Could not parse a valid letter from LLM output, falling back to first option.")
    return opts[0].id

In [34]:
%%time
# 1. Pick a competition
comps = client.competitions.list_all()
print(comps)
COMPETITION_ID = comps[0].id   # or any valid id

# 2. Play the whole game using the LLM
final = client.play_game(COMPETITION_ID, llm_answer_strategy)
print(f"Game over! Earned amount: {final.earned_amount}")

[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Competition(id=0, name='Entertainment', description='Music, Movies, Celebrities and more', max_levels=15, is_infinite=False, created_at='2026-04-12 19:29:31', question_count=843), Competition(id=1, name='Ancient History and Politics', description='The Roman Empire, The Greeks, and more', max_levels=15, is_infinite=False, created_at='2026-04-12 19:29:57', question_count=454), Competition(id=2, name='Science and Nature', description='Chemistry, Biology, Physics and similar subjects', max_levels=15, is_infinite=False, created_at='2026-04-12 19:30:13', question_count=5395), Competition(id=3, name='Maths', description='Mathematics and Statistics from High School and College', max_levels=15, is_infinite=False, created_at='2026-04-12 19:30:22', question_count=390)]


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: <|im_start|>user
You are a very smart and intelligent model that will always answer this question correctly, do not get it wrong for any reason at all
Question: Which of the following best describes the influence of Little Richard's song 'Tutti Frutti' on David Bowie?
Options:
A: It convinced him to become a classical musician
B: It introduced him to the concept of rock and roll
C: It led him to pursue a career in academia
D: It inspired him to become a filmmaker
Answer with just the letter of the correct option (A, B, C, or D).
Answer:<|im_end|>

[LLM raw output] 'B'


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Prompt: <|im_start|>user
You are a very smart and intelligent model that will always answer this question correctly, do not get it wrong for any reason at all
Question: Which of the following best describes the Rolling Stones' sound in the early 1960s?
Options:
A: Gritty and rhythmically driven, defining hard rock
B: Soft and melodic, similar to folk music
C: Uplifting and optimistic, like pop music
D: Lyrical and acoustic, akin to country music
Answer with just the letter of the correct option (A, B, C, or D).
Answer:<|im_end|>

[LLM raw output] 'A'


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: <|im_start|>user
You are a very smart and intelligent model that will always answer this question correctly, do not get it wrong for any reason at all
Question: Which of Hitchcock's films is known for its innovative use of camera techniques, particularly the 'dolly zoom' effect?
Options:
A: North by Northwest
B: Psycho
C: The Birds
D: Vertigo
Answer with just the letter of the correct option (A, B, C, or D).
Answer:<|im_end|>

[LLM raw output] 'D'


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: <|im_start|>user
You are a very smart and intelligent model that will always answer this question correctly, do not get it wrong for any reason at all
Question: Which of the following best describes the production and broadcasting history of LazyTown?
Options:
A: It was first produced as a stage play and later adapted into a television series.
B: It was created by a professional athlete and premiered in 2001.
C: It was produced exclusively for Nickelodeon in the United States.
D: It was originally produced in Icelandic and broadcast in over 180 countries.
Answer with just the letter of the correct option (A, B, C, or D).
Answer:<|im_end|>

[LLM raw output] 'A'
Prompt: <|im_start|>user
You are a very smart and intelligent model that will always answer this question correctly, do not get it wrong for any reason at all
Question: Which instrument did Paul McCartney primarily use to compose the Beatles' hit 'Yesterday'?
Options:
A: Bass guitar
B: Acoustic guitar
C: Piano
D: Drums
An